# Download Data for WWDWA Percentages Map

Downloads county-level data needed for the WWDWA composite score:
- **Female %** — from Census ACS 5-year 2023 (B01001)
- **Median age** — from Census ACS 5-year 2023 (B01002)
- **Total population & White non-Hispanic %** — from Census ACS 5-year 2023 (B03002)
- **Trump 2024 vote %** — from tonmcg GitHub dataset

**Requirements:**
```bash
pip install requests python-dotenv
```

**Setup:**
Create a `.env` file in the project root with:
```
census_api_key=YOUR_API_KEY_HERE
```

In [ ]:
import os
import json
import csv
import io
import requests
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
census_api_key = os.getenv("census_api_key")
if not census_api_key:
    raise ValueError("Missing census_api_key in .env file!")

PUBLIC_DIR = (Path.cwd().parent / "public").resolve()
print(f"Output directory: {PUBLIC_DIR}")

API_URL = "https://api.census.gov/data/2023/acs/acs5"

In [ ]:
# ============================================================
# 1. CENSUS: Demographics (race, sex, median age)
# ============================================================
VARIABLES = {
    "B03002_001E": "total_pop",
    "B03002_003E": "white_non_hisp",
    "B01001_026E": "female_pop",
    "B01002_001E": "median_age",
}

print("Downloading Census county demographics...")
response = requests.get(API_URL, params={
    "get": "NAME," + ",".join(VARIABLES.keys()),
    "for": "county:*",
    "key": census_api_key,
})
response.raise_for_status()
data = response.json()

headers = data[0]
census_map = {}
for row in data[1:]:
    geoid = row[headers.index("state")] + row[headers.index("county")]
    record = {
        "GEOID": geoid,
        "name": row[headers.index("NAME")],
    }
    for var_code, var_name in VARIABLES.items():
        val = row[headers.index(var_code)]
        if var_name == "median_age":
            record[var_name] = float(val) if val and val != "-666666666" else None
        else:
            record[var_name] = int(val) if val and val != "-666666666" else 0
    census_map[geoid] = record

print(f"  ✅ Got {len(census_map)} counties from Census")

In [ ]:
# ============================================================
# 2. ELECTION: 2024 Trump vote percentage by county
# ============================================================
ELECTION_URL = "https://raw.githubusercontent.com/tonmcg/US_County_Level_Election_Results_08-24/master/2024_US_County_Level_Presidential_Results.csv"

print("Downloading 2024 election results...")
response = requests.get(ELECTION_URL)
response.raise_for_status()

reader = csv.DictReader(io.StringIO(response.text))
election_map = {}
for row in reader:
    fips = row["county_fips"].zfill(5)
    try:
        trump_pct = float(row["per_gop"]) * 100
    except (ValueError, KeyError):
        trump_pct = None
    election_map[fips] = trump_pct

print(f"  ✅ Got {len(election_map)} counties from election data")

In [ ]:
# ============================================================
# 3. MERGE & SAVE
# ============================================================
combined = []
matched_election = 0

for geoid, record in census_map.items():
    total_pop = record["total_pop"]
    female_pop = record["female_pop"]
    white_non_hisp = record["white_non_hisp"]
    median_age = record["median_age"]

    female_pct = (female_pop / total_pop * 100) if total_pop > 0 else None
    white_pct = (white_non_hisp / total_pop * 100) if total_pop > 0 else None

    trump_pct = election_map.get(geoid)
    if trump_pct is not None:
        matched_election += 1

    entry = {
        "GEOID": geoid,
        "name": record["name"],
        "total_pop": total_pop,
        "female_pct": round(female_pct, 2) if female_pct is not None else None,
        "median_age": median_age,
        "white_pct": round(white_pct, 2) if white_pct is not None else None,
        "trump_pct": round(trump_pct, 2) if trump_pct is not None else None,
    }
    combined.append(entry)

output_path = PUBLIC_DIR / "US_county_percentages.json"
with open(output_path, "w") as f:
    json.dump(combined, f)

print(f"\n✅ Saved {len(combined)} counties to US_county_percentages.json")
print(f"   Election data matched: {matched_election}/{len(combined)}")
print(f"\nSample record:")
print(json.dumps(combined[0], indent=2))